In [0]:
env = dbutils.widgets.get("environment") if dbutils.widgets.getAll().get("environment") else "dev"
bronzeTablName = f"saleslake_{env}.bronze_{env}.rawProduct"
silverTablName = f"saleslake_{env}.silver_{env}.cleanedProduct"

spark.sql(f"""
INSERT INTO {silverTablName}
SELECT DISTINCT
    CAST(TRIM(product_id) AS INT)             AS product_id,
    UPPER(TRIM(sku))                          AS sku,
    UPPER(TRIM(product_name))                       AS product_name,
    UPPER(TRIM(category))                     AS category,
    UPPER(TRIM(sub_category))                 AS sub_category,
    UPPER(TRIM(brand))                        AS brand,
    UPPER(TRIM(supplier))                     AS supplier,
    CAST(TRIM(unit_cost) AS DECIMAL(18,2))    AS unit_cost,
    CAST(TRIM(list_price) AS DECIMAL(18,2))   AS list_price,
    TO_DATE(TRIM(launch_date), 'yyyy-MM-dd')  AS launch_date,
    UPPER(TRIM(status))                       AS status,
    CURRENT_TIMESTAMP() AS ingest_ts
FROM {bronzeTablName}
WHERE ingest_ts > (
    SELECT COALESCE(MAX(ingest_ts), TO_TIMESTAMP("1990-01-01"))
    FROM {silverTablName}
)
ORDER BY product_id
""")

print(f"Silver load complete for {silverTablName}")
spark.sql(f"SELECT COUNT(*) AS row_count FROM {silverTablName}").display()